In [1]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
import json
from fpdf import FPDF
import pinecone
from langchain.vectorstores import Pinecone
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain.llms import CTransformers

c:\Users\Yashwanth\AppData\Local\Programs\Python\Python310\lib\site-packages\pinecone\index.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
def load_pdf(data):
    loader = DirectoryLoader(data,glob="*.pdf",loader_cls=PyPDFLoader)
    doucuments = loader.load()
    return doucuments

In [23]:
extracted_data = load_pdf('data/')

In [24]:
extracted_data

[Document(page_content='Match: World T20 - T20  \nDate: 2016-03-23  \nVenue: Feroz Shah Kotla, Delhi  \nTeams: Afghanistan vs England  \nToss: England won the toss and decided to bat  \nResult: England won by 15 runs  \nMan of the Match: MM Ali  \n  \nInnings 1:  \n0.1: JJ Roy faced Amir Hamza, scored 1 runs. Team score: 1/0  \n0.2: JM Vince faced Amir Hamza, scored 0 runs. Team score: 1/0  \n0.3: JM Vince faced Amir Hamza, scored 0 runs. Team score: 1/0  \n0.4: JM Vince faced Amir Hamza, scored 0 runs. Team score: 1/0  \n0.5: JM Vince faced Amir Hamza, scored 1 runs. Team score: 2/0  \n0.6: JJ Roy faced Amir Hamza, scored 0 runs. Team score: 2/0  \n1.1: JM Vince faced Shapoor Zadran, scored 1 runs. Team score: 3/0  \n1.2: JJ Roy faced Shapoor Zadran, scored 0 runs. Team score: 3/0  \n1.3: JJ Roy faced Shapoor Zadran, scored 3 runs. Team score: 6/0  \n1.4: JM Vince faced Shapoor Zadran, scored 1 runs. Team score: 7/0  \n1.5: JJ Roy faced Shapoor Zadran, scored 4 runs. Team score: 11/0 

In [4]:
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = 100, chunk_overlap = 10)
    text_chunks = text_splitter.split_documents(extracted_data)

    return text_chunks

In [5]:
text_chunks = text_split(extracted_data)
print("Length of my chunk:",len(text_chunks ))

Length of my chunk: 307


In [6]:
def huggging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [7]:
embeddings = huggging_face_embeddings()

c:\Users\Yashwanth\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [8]:
embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [9]:
API_KEY = "dd3112c0-e58c-40a2-9814-c4163cf4509f"
API_ENV = "gcp-starter"

In [10]:
pinecone.init(api_key=API_KEY,environment=API_ENV)
# index_name = "cricket"
# pinecone.create_index("cricket",dimension=384)


In [11]:
pinecone.list_indexes()

['cricket']

In [12]:
index_name = "cricket"


In [13]:

docsearch=Pinecone.from_texts([t.page_content for t in text_chunks], embeddings, index_name=index_name)

c:\Users\Yashwanth\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\models\bert\modeling_bert.py:435: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


In [14]:
docsearch=Pinecone.from_existing_index(index_name, embeddings)

query = "Which countries played the match?"

docs=docsearch.similarity_search(query,k=3)

print("Result", docs)

Result [Document(page_content='Match: World T20 - T20  \nDate: 2016-03-23  \nVenue: Feroz Shah Kotla, Delhi  \nTeams: Afghanistan vs England  \nUmpires: S Ravi, RJ Tucker  \nToss: England won the toss and decided to bat  \nResult: England won by 15 runs  \nMan of the Match: MM Ali'), Document(page_content='Match: World T20 - T20  \nDate: 2016-03-23  \nVenue: Feroz Shah Kotla, Delhi  \nTeams: Afghanistan vs England  \nUmpires: S Ravi, RJ Tucker  \nToss: England won the toss and decided to bat  \nResult: England won by 15 runs  \nMan of the Match: MM Ali'), Document(page_content='Match: World T20 - T20  \nDate: 2016-03-23  \nVenue: Feroz Shah Kotla, Delhi  \nTeams: Afghanistan vs England  \nUmpires: S Ravi, RJ Tucker  \nToss: England won the toss and decided to bat  \nResult: England won by 15 runs  \nMan of the Match: MM Ali')]


In [15]:
prompt_template="""
Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: {context}
Question: {question}

Only return the helpful answer below and nothing else.
Helpful answer:
"""

In [16]:
PROMPT=PromptTemplate(template=prompt_template, input_variables=["context", "question"])
chain_type_kwargs={"prompt": PROMPT}

In [17]:
llm=CTransformers(model="model/llama-2-7b-chat.ggmlv3.q4_0.bin",
                  model_type="llama",
                  config={'max_new_tokens':512,
                          'temperature':0.8})

In [18]:
qa=RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="stuff", 
    retriever=docsearch.as_retriever(search_kwargs={'k': 2}),
    return_source_documents=True, 
    chain_type_kwargs=chain_type_kwargs)

In [19]:
user_input="What was the man of the match?"
result=qa({"query": user_input})
print("Response : ", result["result"])

c:\Users\Yashwanth\AppData\Local\Programs\Python\Python310\lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(


Response :  MM Ali


In [20]:
user_input="Where did the match happen?"
result=qa({"query": user_input})
print("Response : ", result["result"])

Response :  The match happened at Feroz Shah Kotla in Delhi.


In [25]:
user_input="What was the result??"
result=qa({"query": user_input})
print("Response : ", result["result"])

Response :  The result of Innings 1 was a victory for England, with MM Ali being named Man of the Match.


In [26]:
user_input="What happened in Over 18 Ball 2?"
result=qa({"query": user_input})
print("Response : ", result["result"])

Response :  Over 18 Ball 2 was not played, as the game had already reached its natural conclusion with the home team winning by a margin of 42 runs.


In [27]:
user_input="What happened in Over 12 Ball 3?"
result=qa({"query": user_input})
print("Response : ", result["result"])

Response :  In over 12 ball 3, Asghar Stanikzai faced CJ Jordan and scored 0 runs.


In [28]:
user_input="What happened in Over 14 Ball 5?"
result=qa({"query": user_input})
print("Response : ", result["result"])

Response :  In Over 1 ball 5, Asghar Stanikzai faced CJ Jordan and scored 0 runs.
